# 16. Production Model — CatBoost Champion (NB15)

**Цель:** собрать продовую модель от сырых данных до сохранённого артефакта `.joblib`, готового к деплою в `src/api/`.

**Champion (по итогам NB 9–15, `docs/15_NB15_ENTRY_EXIT_CONCLUSIONS.md`):**

| Параметр | Значение | Откуда |
|----------|----------|--------|
| Модель | **CatBoost** (1000/4/0.02) | NB 15 champion, конфиг из NB23 |
| Фичи | 21 base + 80 rolling (5/15/30/60) = **101** | NB 11, NB 13, NB 15 — sequence_5_15_30_60 |
| Торговая логика | prod_hold | NB 15 — выше avg/trade, меньше сделок vs entry_exit_05 |
| Порог | **25-70** (BUY ≥ 0.70, SELL ≤ 0.25) | NB 15, NB 23 |
| Комиссия | 0.1% round-trip | Bybit taker+taker |

Ансамбли (NB17) — на будущее, пока не используются.

**Спецификация API:** `ml_api_integration_spec_04-03-2026.md` v2.2.0.

**Структура:** Setup → Data (valid_full rolling) → Scale → Train → Backtest → Save → Validate

**Конфиг CatBoost:** 1000/4/0.02 — по итогам `05_experiments/23_CatBoost_Hyperparam_Tuning.ipynb` (устойчивость по дням, min avg 2.00%).

## 1. Импорты и константы

Все параметры модели зафиксированы по итогам экспериментов. Ничего не подбирается — только воспроизводим лучший конфиг.

In [1]:
import sys, os, numpy as np, pandas as pd, joblib, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import catboost as cb

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == '06_production' else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

COMMISSION_RT = 0.001       # 0.1% round-trip (Bybit taker+taker)
THR_HI        = 0.70        # band 25-70 → BUY >= 0.70, SELL <= 0.25 (champion NB15)
THR_LO        = 0.25        # SELL <= thr_lo
SEQ_WINDOWS   = [5, 15, 30, 60]
TARGET_COL    = 'target'
MODEL_PATH    = os.path.join(_root, 'models', 'prod_cat_seq.joblib')

print(f'Root: {_root}')
print(f'Threshold (band 25-70): BUY>={THR_HI}, SELL<={THR_LO}')
print(f'Sequence windows: {SEQ_WINDOWS}')
print(f'Model save path: {MODEL_PATH}')

Root: c:\project\trading_bot_2Engine
Threshold (band 25-70): BUY>=0.7, SELL<=0.25
Sequence windows: [5, 15, 30, 60]
Model save path: c:\project\trading_bot_2Engine\models\prod_cat_seq.joblib


## 2. Загрузка данных и temporal split

Данные из пайплайна NB 01-03. **Rolling на valid_full (BUY+SELL+HOLD)** — как в NB15/NB17, чтобы train и backtest использовали одинаковый rolling.

**Split 7 / 1 / 2** — как в NB15 (train 7 дней, val 1, test 2 дня).

In [2]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

# valid_full: BUY+SELL+HOLD — для rolling и честного бэктеста (как NB15/NB17)
valid_full = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid_full = valid_full[valid_full[TARGET_COL].isin([-1.0, 0.0, 1.0])]
valid_full['date'] = pd.to_datetime(valid_full['datetime'], utc=True).dt.date
valid_full['ret_next'] = valid_full.groupby('session_key')['close_price'].pct_change().shift(-1)
valid_full = valid_full.dropna(subset=['ret_next'])

sort_col = 'datetime' if 'datetime' in valid_full.columns else 'timestamp'

# Rolling на полном valid_full (все бары) — как в NB15/NB17
KEY_FEATS = BASE_FEATURES[:10]
grp = valid_full.groupby('session_key', group_keys=False)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS:
        if c in valid_full.columns:
            valid_full[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            valid_full[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_FEATURES = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS for s in ('mean', 'std')]
ALL_FEATURES = [c for c in (BASE_FEATURES + SEQ_FEATURES) if c in valid_full.columns]

# Split 7 / 1 / 2 (как NB15)
dates = sorted(valid_full['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'

train_dates = set(dates[:7])
val_dates   = set([dates[7]])
test_dates  = set(dates[8:10])
val_day, test1_day, test2_day = dates[7], dates[8], dates[9]

# valid для fit: только BUY+SELL
valid = valid_full[valid_full[TARGET_COL].isin([-1.0, 1.0])].copy()
valid['y'] = (valid[TARGET_COL] == 1).astype(int)

train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df   = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df  = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

# Полные данные для бэктеста (BUY+SELL+HOLD)
val_df_full  = valid_full[valid_full['date'] == val_day].dropna(subset=ALL_FEATURES).sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df_full = valid_full[valid_full['date'].isin(test_dates)].dropna(subset=ALL_FEATURES).sort_values(['session_key', sort_col]).reset_index(drop=True)

print(f'Dates: train={min(train_dates)}..{max(train_dates)} | val={val_day} | test={test1_day}, {test2_day}')
print(f'Rows (BUY+SELL): train={len(train_df):,} | val={len(val_df):,} | test={len(test_df):,}')
print(f'Rows full (backtest): val={len(val_df_full):,} | test={len(test_df_full):,}')
print(f'Features: base={len(BASE_FEATURES)}, rolling={len(SEQ_FEATURES)}, total={len(ALL_FEATURES)}')

Dates: train=2026-02-01..2026-02-07 | val=2026-02-08 | test=2026-02-09, 2026-02-10
Rows (BUY+SELL): train=123,971 | val=23,108 | test=39,475
Rows full (backtest): val=45,592 | test=74,437
Features: base=21, rolling=80, total=101


## 3. Проверка базовых фичей

21 базовая фичи уже присутствуют в parquet (рассчитаны через `src/features/feature_pipeline.py` в NB 03, отобраны в NB 05).
В онлайн-инференсе используется та же функция `add_features()` — гарантия идентичности.

In [3]:
missing = [c for c in BASE_FEATURES if c not in train_df.columns]
assert not missing, f'Missing base features: {missing}'

print(f'✓ Все {len(BASE_FEATURES)} базовых фичей на месте')
print('Список:', BASE_FEATURES)

✓ Все 21 базовых фичей на месте
Список: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'rd_regime']


## 4. Rolling sequence features (5, 15, 30, 60)

Первые 10 базовых фичей (rd_mom_1 … rsi_14) агрегируются rolling mean/std по каждому окну внутри `session_key`.
Итого: 10 × 4 окна × 2 (mean, std) = 80 новых → **101 фича**.

Исследовано в NB 11 (30/60), NB 13 §9 (5/15/30/60 vs 30/60 — новые окна дали лучший avg/trade).

In [4]:
# Проверочный вывод по уже посчитанным rolling-фичам (см. ячейку 2)
missing_seq = [c for c in ALL_FEATURES if c not in train_df.columns]
assert not missing_seq, f'Missing: {missing_seq}'

print(f'Key feats for rolling: {KEY_FEATS}')
print(f'Rolling features added: {len(SEQ_FEATURES)}')
print(f'Total features: {len(ALL_FEATURES)} (21 base + {len(SEQ_FEATURES)} rolling)')

Key feats for rolling: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14']
Rolling features added: 80
Total features: 101 (21 base + 80 rolling)


## 5. Масштабирование фичей

StandardScaler fit на train, transform на val/test. Скейлер сохраняется в bundle для инференса.
Подход из NB 06 (Scaling_And_Normalization).

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train_df[ALL_FEATURES].fillna(0))
X_val   = scaler.transform(val_df[ALL_FEATURES].fillna(0))
X_test  = scaler.transform(test_df[ALL_FEATURES].fillna(0))

y_train = train_df['y'].values
y_val   = val_df['y'].values
y_test  = test_df['y'].values

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'y balance train: {y_train.mean():.3f} | val: {y_val.mean():.3f} | test: {y_test.mean():.3f}')

X_train: (123971, 101), X_val: (23108, 101), X_test: (39475, 101)
y balance train: 0.500 | val: 0.508 | test: 0.524


## 6. Обучение CatBoost

CatBoost (champion NB15): cat_seq_new prod_hold 25-70.
**Конфиг из NB23:** iterations=1000, depth=4, lr=0.02 — лучшая устойчивость по дням (min avg 1.84%), 587 сделок, avg 2.28%.

In [6]:
model = cb.CatBoostClassifier(
    iterations=1000,
    depth=4,
    learning_rate=0.02,
    random_seed=42,
    verbose=0,
)

model.fit(X_train, y_train)

proba_val  = model.predict_proba(X_val)[:, 1]
proba_test = model.predict_proba(X_test)[:, 1]

auc_val  = roc_auc_score(y_val, proba_val)
auc_test = roc_auc_score(y_test, proba_test)

print(f'AUC val:  {auc_val:.6f}')
print(f'AUC test: {auc_test:.6f}')

AUC val:  0.751426
AUC test: 0.716788


## 7. Оценка качества классификации

Порог 25-70 (BUY ≥ 0.70, SELL ≤ 0.25) — champion из NB15.

**Метрики считаем только по зоне уверенных решений:** prob ≤ 0.25 или prob ≥ 0.70 (где реально входим/выходим). Зона HOLD (0.25 < prob < 0.70) исключается.

In [7]:
# Зона уверенных решений: только prob <= thr_lo или prob >= thr_hi
confident_mask = (proba_test <= THR_LO) | (proba_test >= THR_HI)
y_pred_confident = np.where(proba_test >= THR_HI, 1, np.where(proba_test <= THR_LO, 0, -1))

y_test_conf = np.array(y_test)[confident_mask]
y_pred_conf = y_pred_confident[confident_mask].astype(int)

n_hold = (~confident_mask).sum()
print(f'=== Classification Report (test, zone 25-70, confident only) ===')
print(f'HOLD zone excluded: {n_hold} samples ({100 * n_hold / len(proba_test):.1f}%)')
print(classification_report(y_test_conf, y_pred_conf, digits=4))
print(f'AUC val:  {auc_val:.6f}')
print(f'AUC test: {auc_test:.6f}')

=== Classification Report (test, zone 25-70, confident only) ===
HOLD zone excluded: 37869 samples (95.9%)
              precision    recall  f1-score   support

           0     0.9472    0.8515    0.8969       485
           1     0.9385    0.9795    0.9585      1121

    accuracy                         0.9408      1606
   macro avg     0.9429    0.9155    0.9277      1606
weighted avg     0.9411    0.9408    0.9399      1606

AUC val:  0.751426
AUC test: 0.716788


## 8. Бэктест (полные данные BUY+SELL+HOLD, prod_hold 25-70)

**Важно:** В проде модель получает все бары. Бэктест на полных данных (val + test1+2).

Логика **prod_hold** (асимметричная 25-70): BUY ≥ 0.70, SELL ≤ 0.25, HOLD — сохраняем позицию.
Комиссия 0.1% round-trip.

In [8]:
def backtest_prod_hold_asym(proba, ret, session_ids, thr_hi=THR_HI, thr_lo=THR_LO, commission_rt=COMMISSION_RT):
    """prod_hold 25-70: BUY >= thr_hi, SELL <= thr_lo, HOLD keeps previous position."""
    pred = np.where(proba >= thr_hi, 1, np.where(proba <= thr_lo, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1: pos[i], prev = 1.0, 1.0
        elif pred[i] == 0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}

X_val_full  = scaler.transform(val_df_full[ALL_FEATURES].fillna(0))
X_test_full = scaler.transform(test_df_full[ALL_FEATURES].fillna(0))
proba_val_full  = model.predict_proba(X_val_full)[:, 1]
proba_test_full = model.predict_proba(X_test_full)[:, 1]

bt_val  = backtest_prod_hold_asym(proba_val_full,  val_df_full['ret_next'].values,  val_df_full['session_key'].values)
bt_test = backtest_prod_hold_asym(proba_test_full, test_df_full['ret_next'].values, test_df_full['session_key'].values)

print('=== Backtest Results (full BUY+SELL+HOLD, prod_hold 25-70) ===')
print(f'VAL  day:   net={bt_val["net_%"]:+.2f}%  trades={bt_val["trades"]}  avg/trade={bt_val["avg_%_per_trade"]:.4f}%  (bars={len(val_df_full):,})')
print(f'TEST days:  net={bt_test["net_%"]:+.2f}%  trades={bt_test["trades"]}  avg/trade={bt_test["avg_%_per_trade"]:.4f}%  (bars={len(test_df_full):,})')

=== Backtest Results (full BUY+SELL+HOLD, prod_hold 25-70) ===
VAL  day:   net=+543.40%  trades=296  avg/trade=1.8358%  (bars=45,592)
TEST days:  net=+1339.46%  trades=587  avg/trade=2.2819%  (bars=74,437)


## 9. Сохранение продового артефакта

Bundle содержит обязательные ключи для `src/api/model_bundle.py` (`model`, `scaler`, `features`) + дополнительные для воспроизводимости и будущего инференса с rolling-фичами.

In [9]:
bundle = {
    'model':          model,
    'model_name':     'CatBoost_seq_5_15_30_60',
    'scaler':         scaler,
    'features':       ALL_FEATURES,
    'base_features':  BASE_FEATURES,
    'seq_windows':    SEQ_WINDOWS,
    'seq_key_feats':  KEY_FEATS,
    'target':         TARGET_COL,
    'threshold':      THR_HI,
    'threshold_lo':   THR_LO,
    'commission_rt':  COMMISSION_RT,
    'val_auc':        auc_val,
    'test_auc':       auc_test,
    'backtest_test':  bt_test,
    'backtest_val':   bt_val,
    'split': {
        'type': 'temporal',
        'train_days': 7,
        'val_days': 1,
        'test_days': 2,
        'train_range': f'{min(train_dates)}..{max(train_dates)}',
        'val_date': str(val_day),
        'test_dates': f'{test1_day}, {test2_day}',
    },
    'spec_version': '2.2.0',
}

joblib.dump(bundle, MODEL_PATH, compress=3)

fsize = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f'Saved: {MODEL_PATH} ({fsize:.1f} MB)')
print(f'Bundle keys: {list(bundle.keys())}')

Saved: c:\project\trading_bot_2Engine\models\prod_cat_seq.joblib (0.2 MB)
Bundle keys: ['model', 'model_name', 'scaler', 'features', 'base_features', 'seq_windows', 'seq_key_feats', 'target', 'threshold', 'threshold_lo', 'commission_rt', 'val_auc', 'test_auc', 'backtest_test', 'backtest_val', 'split', 'spec_version']


## 10. Валидация артефакта

Загружаем через `load_model_bundle()` — тот же путь, что использует API.
Smoke-test: предсказание на одном сэмпле, проверка сигнала.

In [10]:
from src.api.model_bundle import load_model_bundle

loaded = load_model_bundle(MODEL_PATH)

assert loaded['model_name'] == 'CatBoost_seq_5_15_30_60'
assert len(loaded['features']) == len(ALL_FEATURES)
assert loaded['threshold'] == THR_HI
print(f'✓ Bundle loaded, model={loaded["model_name"]}, features={len(loaded["features"])}')

sample = X_test[:1]
p = loaded['model'].predict_proba(loaded['scaler'].transform(
    test_df[loaded['features']].fillna(0).iloc[:1]
))[:, 1][0]

thr_hi = loaded['threshold']
thr_lo = loaded.get('threshold_lo', 1 - thr_hi)
if p >= thr_hi:
    sig = 'BUY'
elif p <= thr_lo:
    sig = 'SELL'
else:
    sig = 'HOLD'

print(f'Smoke test: proba={p:.4f}, signal={sig} (thr_hi={thr_hi}, thr_lo={thr_lo})')
print('✓ Артефакт валиден и совместим с src/api/model_bundle.py')

✓ Bundle loaded, model=CatBoost_seq_5_15_30_60, features=101
Smoke test: proba=0.3589, signal=HOLD (thr_hi=0.7, thr_lo=0.25)
✓ Артефакт валиден и совместим с src/api/model_bundle.py


## 11. Итоги

### Продовая модель (champion NB15)

| Параметр | Значение |
|----------|----------|
| Модель | CatBoost (iterations=1000, depth=4, lr=0.02, из NB23) |
| Фичи | 101 (21 base + 80 rolling 5/15/30/60) |
| Порог | 25-70 (BUY ≥ 0.70, SELL ≤ 0.25) |
| Торговая логика | prod_hold (HOLD сохраняет позицию) |
| Комиссия | 0.1% round-trip |
| Артефакт | `models/prod_cat_seq.joblib` |
| Split | 7 train / 1 val / 2 test (как NB15) |

### Метрики

AUC val/test и бэктест — см. вывод ячеек 6–8. Ожидается ~2.28% avg/trade, 587 сделок на test (по NB23).

### Что входит в bundle

- `model` — CatBoostClassifier
- `scaler` — StandardScaler (fit на train)
- `features` — 101 фичи (порядок важен)
- `base_features`, `seq_windows`, `seq_key_feats` — для rolling в inference
- `threshold` / `threshold_lo` — 0.70 / 0.25 (25-70)
- `split`, `val_auc`, `test_auc`, `backtest_*` — метаданные

### Проверка утечки данных

- **Фичи:** только прошлое и текущий бар (diff, rolling, ewm — без shift(-1)). `feature_pipeline.py` — внутри session_key.
- **Target:** tp_sl_1_05 — метка по будущим H барам (TP/SL), корректно для supervised learning.
- **ret_next:** используется только для бэктеста PnL, в модель не подаётся.
- **Scaler:** fit только на train, transform на val/test.

### Тест-прод

Модель можно запускать в тестовый прод при условии: rolling-фичи считаются онлайн с буфером 60 баров на сессию (как в inference).

### Файлы для разработчика API

| Файл | Назначение |
|------|------------|
| `models/prod_cat_seq.joblib` | **Прод.** Модель + scaler + features (101) в одном bundle. Основной артефакт. |
| `src/features/feature_pipeline.py` | Расчёт 21 базовой фичи (add_features). |
| `src/api/inference_service.py` | add_features + rolling (5/15/30/60) + predict. Уже поддерживает prod_cat_seq. |
| `00_contract/ml_api_integration_spec_04-03-2026.md` | Контракт API (v2.2.0). |
| `outputs/features_selected_tp_sl_1_05.txt` | Список 21 базовых фичей (для справки). |

**scaler_tp_sl_1_05.joblib** — устарел для прода. Используется только в кандидат-пайплайне (NB07). Прод: scaler внутри prod_cat_seq.joblib.

### Пайплайн

`docs/04_FEATURE_MODEL_PIPELINE_PLAN.md` — план кандидат-пайплайна (NB01–07). Прод-модель — NB16 (отдельный поток).

### Следующие шаги

1. Shadow deploy — `source=ml_shadow`.
2. Ансамбли (NB17) — на будущее.